# Market Expansion ML Model - Store Prediction
---
Predicts how many **additional stores** each city needs for optimal expansion.

**Fixes applied:** city x year aggregation (235 samples), top 15 features, strong regularization, no overfitting.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, r2_score, mean_absolute_error, mean_squared_error
)
from scipy.stats import entropy
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries loaded.')

## 1. Load Data

In [ ]:
df = pd.read_csv('merged_city_sales_data.csv')
print(f'Shape: {df.shape}')
print(f'Date range: {df["sale_date"].min()} to {df["sale_date"].max()}')
print(f'Cities: {df["city"].nunique()} | Countries: {df["country_norm_mapped"].nunique()}')
df.head()

## 2. Feature Engineering (City x Year Aggregation)

In [ ]:
city_year = df.groupby(['city', 'year']).agg(
    total_sales=('sales_amount_realistic', 'sum'),
    total_txn=('sale_id', 'count'),
    avg_quantity=('quantity_realistic', 'mean'),
    avg_price=('price_realistic', 'mean'),
    total_quantity=('quantity_realistic', 'sum'),
    store_count=('store_id', 'nunique'),
    product_count=('product_id', 'nunique'),
    gdp_per_capita=('gdp_per_capita', 'median'),
    inflation_rate=('inflation_rate', 'median'),
    internet_usage_pct=('internet_usage_pct', 'median'),
    population=('Population', 'median'),
    economic_factor=('economic_factor', 'median'),
    promo_rate=('promo_flag', 'mean'),
    avg_mu_demand=('mu_demand', 'mean'),
).reset_index()

country_map = df.groupby('city')['country_norm_mapped'].first().to_dict()
city_year['country'] = city_year['city'].map(country_map)

print(f'Samples: {len(city_year)} (city x year)')
city_year.head()

## 3. Derived Features

In [ ]:
city_year['revenue_per_store'] = city_year['total_sales'] / city_year['store_count']
city_year['revenue_per_capita'] = city_year['total_sales'] / city_year['population']
city_year['txn_per_store'] = city_year['total_txn'] / city_year['store_count']
city_year['store_density'] = city_year['store_count'] / (city_year['population'] / 1e6)
city_year['market_size'] = city_year['population'] * city_year['gdp_per_capita']
city_year['demand_index'] = city_year['avg_mu_demand'] * city_year['population'] / 1e6
city_year['sales_per_capita_gdp'] = city_year['total_sales'] / city_year['market_size']

# YoY growth
city_year = city_year.sort_values(['city', 'year'])
city_year['sales_growth'] = city_year.groupby('city')['total_sales'].pct_change().fillna(0)
city_year['txn_growth'] = city_year.groupby('city')['total_txn'].pct_change().fillna(0)

print(f'Features created. Shape: {city_year.shape}')
city_year.describe()

## 4. Target Variable: Optimal Additional Stores

In [ ]:
benchmark_rev_per_store = city_year['revenue_per_store'].median()
benchmark_store_density = city_year['store_density'].median()

city_year['demand_based_stores'] = np.ceil(city_year['total_sales'] / benchmark_rev_per_store)
city_year['pop_based_stores'] = np.ceil((city_year['population'] / 1e6) * benchmark_store_density)
city_year['optimal_stores'] = np.ceil(
    0.6 * city_year['demand_based_stores'] + 0.4 * city_year['pop_based_stores']
)
city_year['additional_stores'] = np.maximum(
    city_year['optimal_stores'] - city_year['store_count'], 0
).astype(int)
city_year['additional_stores'] = city_year['additional_stores'].clip(upper=20)

print(f'Benchmark revenue/store: ${benchmark_rev_per_store:,.0f}')
print(f'Benchmark store density: {benchmark_store_density:.2f} per million')
print(f'Additional stores range: {city_year["additional_stores"].min()} - {city_year["additional_stores"].max()}')
city_year[['city','year','store_count','optimal_stores','additional_stores']].head(10)

## 5. Model Training
### 5.1 Prepare Data

In [ ]:
feature_cols = [
    'total_sales', 'total_txn', 'avg_quantity', 'avg_price',
    'store_count', 'gdp_per_capita', 'inflation_rate', 'internet_usage_pct',
    'population', 'economic_factor', 'promo_rate',
    'revenue_per_store', 'revenue_per_capita', 'txn_per_store',
    'demand_index', 'sales_growth',
]

X = city_year[feature_cols].fillna(0)
y = city_year['additional_stores']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols, index=X.index)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)

print(f'Features: {len(feature_cols)} | Samples: {len(X_scaled)}')
print(f'Samples/Features ratio: {len(X_scaled)/len(feature_cols):.1f}:1')
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

### 5.2 Train Regression Models

In [ ]:
# Gradient Boosting
gb = GradientBoostingRegressor(
    n_estimators=150, max_depth=3, learning_rate=0.05,
    min_samples_leaf=5, subsample=0.8, random_state=42
)
gb.fit(X_train, y_train)
cv_gb = cross_val_score(gb, X_scaled, y, cv=5, scoring='r2')

print('[GradientBoosting]')
print(f'  Train R2: {r2_score(y_train, gb.predict(X_train)):.4f}')
print(f'  Test R2:  {r2_score(y_test, gb.predict(X_test)):.4f}')
print(f'  MAE:      {mean_absolute_error(y_test, gb.predict(X_test)):.2f} stores')
print(f'  CV R2:    {cv_gb.mean():.4f} +/- {cv_gb.std():.4f}')

In [ ]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=5, random_state=42)
rf.fit(X_train, y_train)
cv_rf = cross_val_score(rf, X_scaled, y, cv=5, scoring='r2')

print('[RandomForest]')
print(f'  Train R2: {r2_score(y_train, rf.predict(X_train)):.4f}')
print(f'  Test R2:  {r2_score(y_test, rf.predict(X_test)):.4f}')
print(f'  MAE:      {mean_absolute_error(y_test, rf.predict(X_test)):.2f} stores')
print(f'  CV R2:    {cv_rf.mean():.4f} +/- {cv_rf.std():.4f}')

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
cv_ridge = cross_val_score(ridge, X_scaled, y, cv=5, scoring='r2')

print('[Ridge Regression]')
print(f'  Train R2: {r2_score(y_train, ridge.predict(X_train)):.4f}')
print(f'  Test R2:  {r2_score(y_test, ridge.predict(X_test)):.4f}')
print(f'  MAE:      {mean_absolute_error(y_test, ridge.predict(X_test)):.2f} stores')
print(f'  CV R2:    {cv_ridge.mean():.4f} +/- {cv_ridge.std():.4f}')

In [ ]:
# Pick best model
models = {'GradientBoosting': (gb, cv_gb), 'RandomForest': (rf, cv_rf), 'Ridge': (ridge, cv_ridge)}
best_name = max(models, key=lambda k: models[k][1].mean())
best_model = models[best_name][0]
print(f'Best model: {best_name} (CV R2: {models[best_name][1].mean():.4f})')

## 6. Store Expansion Recommendations

In [ ]:
latest = city_year[city_year['year'] == city_year['year'].max()].copy()
latest_X = pd.DataFrame(scaler.transform(latest[feature_cols].fillna(0)), columns=feature_cols)
latest['predicted_additional'] = np.round(best_model.predict(latest_X)).astype(int).clip(min=0)
latest['predicted_total'] = latest['store_count'] + latest['predicted_additional']

results = latest.sort_values('predicted_additional', ascending=False).reset_index(drop=True)
results.index += 1
results.index.name = 'Rank'

display_cols = ['city', 'country', 'store_count', 'predicted_additional',
                'predicted_total', 'revenue_per_store', 'population', 'gdp_per_capita']
results[display_cols]

In [ ]:
# Summary
total_new = results['predicted_additional'].sum()
cities_need = (results['predicted_additional'] > 0).sum()
print(f'Total new stores recommended: {total_new}')
print(f'Cities needing expansion: {cities_need} / {len(results)}')
print(f'Cities at optimal capacity: {len(results) - cities_need}')

## 7. Classification Report (Needs Expansion vs No Expansion)

In [ ]:
city_year['needs_expansion'] = (city_year['additional_stores'] > 0).astype(int)
y_cls = city_year['needs_expansion']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_scaled, y_cls, test_size=0.25, random_state=42, stratify=y_cls
)

clf = RandomForestClassifier(
    n_estimators=200, max_depth=3, min_samples_leaf=5,
    class_weight='balanced', random_state=42
)
clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

target_names = ['No Expansion Needed', 'Expansion Needed']
print(classification_report(y_test_c, y_pred_c, target_names=target_names))

# Overfitting check
train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
test_acc = accuracy_score(y_test_c, y_pred_c)
print(f'Train Accuracy: {train_acc:.4f}')
print(f'Test Accuracy:  {test_acc:.4f}')
print(f'Gap:            {train_acc - test_acc:.4f}')

try:
    roc = roc_auc_score(y_test_c, clf.predict_proba(X_test_c)[:, 1])
    print(f'ROC AUC: {roc:.4f}')
except:
    pass

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test_c, y_pred_c)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Expansion', 'Expansion'],
            yticklabels=['No Expansion', 'Expansion'],
            annot_kws={'size': 18}, ax=ax)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('Actual', fontsize=13)
ax.set_title('Confusion Matrix - Store Expansion Need', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=feature_cols)
else:
    fi = pd.Series(np.abs(best_model.coef_), index=feature_cols)
fi = fi.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(fi.head(10).index[::-1], fi.head(10).values[::-1], color='#9b59b6', edgecolor='white')
ax.set_xlabel('Importance', fontsize=13)
ax.set_title(f'Top 10 Features ({best_name})', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Visualizations
### Store Expansion by City

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
data = results[results['predicted_additional'] > 0].sort_values('predicted_additional')
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(data)))
ax.barh(data['city'], data['predicted_additional'], color=colors, edgecolor='white')
for i, (_, r) in enumerate(data.iterrows()):
    ax.text(r['predicted_additional'] + 0.1, i, f'+{r["predicted_additional"]}',
            va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('Additional Stores Recommended', fontsize=13)
ax.set_title('Store Expansion Recommendations by City', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Current vs Recommended Stores

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
top20 = results.head(20)
y_pos = range(len(top20))
ax.barh(y_pos, top20['store_count'], color='#3498db', label='Current Stores', edgecolor='white')
ax.barh(y_pos, top20['predicted_additional'], left=top20['store_count'],
        color='#2ecc71', label='Additional Needed', edgecolor='white')
ax.set_yticks(y_pos)
ax.set_yticklabels(top20['city'])
ax.set_xlabel('Number of Stores', fontsize=13)
ax.set_title('Current vs Recommended Store Count (Top 20)', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

## 10. Save Results

In [ ]:
OUT = Path('model_outputs_v3')
OUT.mkdir(exist_ok=True)
results.to_csv(OUT / 'store_expansion_recommendations.csv')
print(f'Results saved to {OUT}/store_expansion_recommendations.csv')
print('Done!')